In [ ]:
# ============================================================
# Generate modern Ukrainian variants — direct API calls
# Checkpoint-safe: resumes across model switches by deduplicating
# on sentence TEXT (not row index), loading ALL checkpoint files.
# ============================================================

!pip -q install openai pandas tqdm

import os
import json
import re
import time
from pathlib import Path

import pandas as pd
from tqdm.auto import tqdm
from openai import OpenAI

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    raise ValueError("Set OPENAI_API_KEY in your environment or .env file")

client = OpenAI(api_key=OPENAI_API_KEY)

# ── Config ──────────────────────────────────────────────────
MODEL          = "gpt-5.4-mini"   # fast & cheap; quality is fine for paraphrase
MAX_ROWS       = None
N_VARIANTS     = 3
INPUT_CSV      = "../data/sad_sentences_filtered.csv"   # 600 curated sentences
OUTPUT_CSV     = "../data/modernized_training_pairs_flat.csv"
CHECKPOINT_DIR = Path("generation_checkpoints")

CHECKPOINT_DIR.mkdir(exist_ok=True)

# New checkpoint file for this model (old gpt-5.5 file is kept untouched)
safe_model_name = MODEL.replace(".", "_").replace("-", "_")
ckpt_file = CHECKPOINT_DIR / f"pairs_checkpoint_{safe_model_name}_{N_VARIANTS}variants.jsonl"

# ── Prompts ─────────────────────────────────────────────────
# Goal: generate MODERN INPUTS for future fine-tuning.
# The original classical sentence stays as the TARGET.

SYSTEM_PROMPT = """
You rewrite old/classical Ukrainian excerpts into natural modern Ukrainian for a supervised fine-tuning dataset.

The modern rewrites will be INPUTS. The original classical excerpt will be the TARGET.
So the rewrites must sound modern, natural, and non-classical.

Rules:
- Preserve the main meaning and emotional intensity.
- Use natural Ukrainian a person in 2026 could write in social networks, messages, etc.
- Remove archaic words and old-fashioned syntax.
- Do NOT imitate classical style.
- Do NOT write Russian, English, or mixed Ukrainian-English.
- Do NOT explain.
- Return only valid JSON.

Examples:

Original: "Тяжко мені на світі жити, немає мені радості."
Good modern variants:
[
  "Мені дуже важко жити, і я майже не відчуваю радості.",
  "Останнім часом мені так важко, що нічого вже не радує.",
  "Я почуваюся вигорівшою і не бачу нічого хорошого навколо."
]
Bad variants:
[
  "Ой тяжко мені, доле моя, жити на білому світі.",
  "My life is very hard and sad.",
  "Мені one і я sad."
]

Original: "Покинута всіма, вона сиділа сама й гірко плакала."
Good modern variants:
[
  "Вона почувалася покинутою всіма і тихо плакала на самоті.",
  "Їй здавалося, що вона нікому не потрібна, і від цього було дуже боляче.",
  "Вона залишилася сама і не могла стримати сліз."
]
Bad variants:
[
  "Вона, мов сирота безталанна, ридала над своєю недолею.",
  "She was lonely and cried bitterly.",
  "Вона була lonely і дуже sad."
]
""".strip()


def user_prompt(classic_text: str, n: int = N_VARIANTS) -> str:
    return f"""
Classical Ukrainian excerpt:

{classic_text}

Generate {n} different modern Ukrainian variants.
They should be suitable as realistic user inputs for a style-transfer model.

Return JSON exactly in this shape:
{{
  "keep": true,
  "modern_variants": ["...", "...", "...", "...", "..."]
}}

If the excerpt is not useful for emotional style-transfer training, return:
{{
  "keep": false,
  "modern_variants": []
}}
""".strip()

# ── Structured JSON schema ──────────────────────────────────
# No emotion labels: simpler and less noisy.

MODERNIZATION_SCHEMA = {
    "name": "modernization_result",
    "strict": True,
    "schema": {
        "type": "object",
        "additionalProperties": False,
        "properties": {
            "keep": {"type": "boolean"},
            "modern_variants": {
                "type": "array",
                "minItems": 0,
                "maxItems": N_VARIANTS,
                "items": {"type": "string"}
            }
        },
        "required": ["keep", "modern_variants"]
    }
}

# ── Load sentences ───────────────────────────────────────────
try:
    df
    print(f"Using df from memory ({len(df)} rows)")
except NameError:
    df = pd.read_csv(INPUT_CSV)
    print(f"Loaded {len(df)} sentences from {INPUT_CSV}")

assert "sentence" in df.columns, "Expected a column named 'sentence'."

df = df.dropna(subset=["sentence"]).drop_duplicates("sentence").reset_index(drop=True)
df_work = df.head(MAX_ROWS).copy() if MAX_ROWS else df.copy()
print(f"Processing {len(df_work)} sentences x {N_VARIANTS} variants = up to {len(df_work) * N_VARIANTS} pairs")
print(f"Model: {MODEL}")
print(f"Checkpoint: {ckpt_file}")

# ── Resume from ALL checkpoint files (works across model switches) ──────────
# Dedup by sentence text, not row index — safe when switching CSV files too.
all_pairs = []
done_sentences = set()

for ckpt in sorted(CHECKPOINT_DIR.glob("pairs_checkpoint_*.jsonl")):
    with open(ckpt, encoding="utf-8") as fh:
        for line in fh:
            try:
                pair = json.loads(line)
                all_pairs.append(pair)
                done_sentences.add(pair["classic_target"])
            except json.JSONDecodeError:
                pass
    print(f"  loaded {ckpt.name}")

print(f"Total pairs loaded from all checkpoints: {len(all_pairs)}")
print(f"Unique sentences already processed: {len(done_sentences)}")

rows_todo = df_work[~df_work["sentence"].isin(done_sentences)]
print(f"Rows left to process: {len(rows_todo)}")

# ── Helpers ─────────────────────────────────────────────────
LATIN_RE = re.compile(r"[A-Za-z]")
WHITESPACE_RE = re.compile(r"\s+")


def clean_text(x: str) -> str:
    return WHITESPACE_RE.sub(" ", str(x)).strip()


def is_bad_modern_variant(text: str) -> bool:
    """Light post-filter against common generation mistakes."""
    t = clean_text(text)
    low = t.lower()

    if len(t) < 8:
        return True

    if LATIN_RE.search(t):
        return True

    # These are target/classical-style markers; modern input should not sound like this.
    too_classical = [
        "ой ", "доле моя", "недоле", "безталанна", "безталанний",
        "на білому світі", "лиха доля", "тяжко мені"
    ]
    if any(marker in low for marker in too_classical):
        return True

    return False

# ── API call with retry ──────────────────────────────────────
def call_api(sentence: str, retries: int = 3):
    for attempt in range(retries):
        try:
            resp = client.chat.completions.create(
                model=MODEL,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": user_prompt(sentence)},
                ],
                temperature=1,
                max_completion_tokens=900,
                response_format={
                    "type": "json_schema",
                    "json_schema": MODERNIZATION_SCHEMA,
                },
            )
            content = resp.choices[0].message.content
            return json.loads(content)
        except Exception as e:
            print(f"  attempt {attempt + 1} failed: {e}")
            time.sleep(2 ** attempt)
    return None

# ── Main generation loop ─────────────────────────────────────
with open(ckpt_file, "a", encoding="utf-8") as ckpt_f:
    for row_idx, row in tqdm(rows_todo.iterrows(), total=len(rows_todo), desc="Generating"):
        sentence = clean_text(row["sentence"])
        result = call_api(sentence)

        if result is None or not result.get("keep", True):
            continue

        variants = result.get("modern_variants", [])[:N_VARIANTS]

        for i, variant in enumerate(variants):
            variant = clean_text(variant)
            if is_bad_modern_variant(variant):
                continue

            pair = {
                    "model":          MODEL,
                    "classic_target": sentence,
                    "modern_input":   variant,
                    "source_file":    str(row.get("source_file", "")),
                    "variant_id":     i,
                }

            all_pairs.append(pair)
            ckpt_f.write(json.dumps(pair, ensure_ascii=False) + "\n")
            ckpt_f.flush()

        time.sleep(0.05)

# ── Save flat CSV ────────────────────────────────────────────
flat_df = pd.DataFrame(all_pairs)
flat_df = flat_df.drop_duplicates(subset=["classic_target", "modern_input"]).reset_index(drop=True)
flat_df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8")

print(f"\nDone! {len(flat_df)} training pairs saved to {OUTPUT_CSV}")
if len(flat_df):
    print(flat_df[["classic_target", "modern_input", "model"]].head(8).to_string())



In [ ]:
# ============================================================
# Inspect results and build SFT training files
# ============================================================

import json
import uuid
import random
from pathlib import Path

import pandas as pd
from IPython.display import display

OUTPUT_CSV = "../data/modernized_training_pairs_flat.csv"

flat_df = pd.read_csv(OUTPUT_CSV)
flat_df = flat_df.dropna(subset=["modern_input", "classic_target"]).drop_duplicates(
    subset=["modern_input", "classic_target"]
).reset_index(drop=True)

print(f"Total pairs: {len(flat_df)}")
print(f"Unique classical targets: {flat_df['classic_target'].nunique()}")
print(f"Models used:")
print(flat_df["model"].value_counts().to_string())
print()

display(flat_df[["classic_target", "modern_input", "model"]].head(10))

# ── master_dataset.jsonl ─────────────────────────────────────
Path("data/processed").mkdir(parents=True, exist_ok=True)
master_path = Path("data/processed/master_dataset.jsonl")

with open(master_path, "w", encoding="utf-8") as f:
    for _, row in flat_df.iterrows():
        record = {
            "id":             f"plug_{str(uuid.uuid4())[:8]}_v{int(row['variant_id'])}",
            "classic_target": str(row["classic_target"]),
            "modern_input":   str(row["modern_input"]),
            "source_file":    str(row.get("source_file", "")),
            "source_dataset": "PluG",
            "generation_model": str(row.get("model", "")),
            "quality_score":  4,
            "split":          "train",
        }
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

print(f"master_dataset.jsonl saved — {len(flat_df)} rows")

# ── SFT chat format splits ───────────────────────────────────
Path("data/sft").mkdir(parents=True, exist_ok=True)

INSTRUCTION = "Перепиши сучасний український текст у стилі української класичної літератури."

pairs = flat_df.to_dict("records")
random.seed(42)
random.shuffle(pairs)

n       = len(pairs)
n_train = int(n * 0.80)
n_val   = int(n * 0.10)

splits  = {
    "train": pairs[:n_train],
    "val":   pairs[n_train: n_train + n_val],
    "test":  pairs[n_train + n_val:],
}

for split_name, rows in splits.items():
    out = Path(f"data/sft/{split_name}.jsonl")
    with open(out, "w", encoding="utf-8") as f:
        for r in rows:
            msg = {
                "messages": [
                    {
                        "role": "user",
                        "content": f"{INSTRUCTION}\n\nТекст: {r['modern_input']}"
                    },
                    {
                        "role": "assistant",
                        "content": r["classic_target"]
                    },
                ]
            }
            f.write(json.dumps(msg, ensure_ascii=False) + "\n")

    print(f"  {split_name}: {len(rows)} examples -> data/sft/{split_name}.jsonl")

print("\nSFT dataset ready.")
print("Direction: modern Ukrainian input → authentic classical Ukrainian target")

In [ ]:
pd.set_option('display.max_colwidth', None)
display(flat_df[["classic_target", "modern_input"]].head(10))

In [ ]:
flat_df.shape